# FX Trading Bot (EUR/USD)

<img src= "images/Candlestick_chart.svg" alt="EUR/USD chart" width="800">

* A EUR/USD trading bot using various quant and technical strategies layered together for the Asia/LDN/NY sessions

***
***

## 0) Notebook Dependencies

In [11]:
# Core Dependencies
import os
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
sns.set_style('whitegrid')
import warnings
warnings.filterwarnings('ignore')


# Financial Data and analysis Dependencies
import talib as ta
import yfinance as yf

    #ticker = yf.Ticker("EURUSD=X")
    #eurusd = yf.download(tickers=ticker, interval="5m", period="2y")

# Machine learning dependencies
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans, AgglomerativeClustering, FeatureAgglomeration
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from xgboost import XGBClassifier
from hdbscan import HDBSCAN


# Neural network dependencies

from pytorch_tcn import TCN
import hmmlearn as hmm

# Eveluation and hypertuning dependencies
from sklearn.metrics import classification_report, accuracy_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

import mlflow
import backtrader
import optuna

# Deployment dependencies

import docker

#!pip install ipywidgets ----for the ml interface to work

***
***

## 1) Business Understanding

### 1.1 Problem Definition.


#### Business Objective

Design and validate a **systematic, ML‑assisted EUR/USD trading framework** that:

- Starts with a **small trading account (~$200)**.
- Focuses initially on **intraday / day‑trading horizons**.
- Prioritizes **capital protection, low volatility, and consistent returns** over “max profit”.
- Is **fully systematic** (no discretionary/manual chart reading).
- Can **scale over time** into:
  - multi‑session trading (Asia / London / NY),
  - swing trading,
  - positional (macro‑driven) trading,
  - and eventually a **live automated trading bot**.

The primary business problem is:  
> “Can we build an objectively testable, risk‑controlled EUR/USD strategy driven by quant & ML methods that is viable for a small account and realistically scalable?”



#### What We Are NOT Doing

- No discretionary/manual “gut feeling” trades.
- No overfitted backtests with unrealistic assumptions.
- No expectation of getting rich from a $200 account.
- No trying to predict exact future prices tick‑by‑tick.
- No using ML as a black box without understanding the trading context.



#### End Products

This project aims to produce:

- A clean, feature‑engineered **EUR/USD intraday dataset**.
- A set of **ML models** (starting from simple baselines → more advanced models).
- A **strategy logic layer** that converts model outputs into trade decisions (entries/exits).
- A **risk management module** that keeps drawdowns acceptable for a small account.
- A **deployment plan** for an automated trading bot (running via broker API).

***

### 1.2 Business Constraints



#### Capital Constraints

- Starting capital: approximately **$200**.
- Position sizes must be **micro‑lots (0.01)** or equivalent.
- Maximum risk per trade: **0.25–0.5%** of account.
- Strategy must survive inevitable losing streaks without blowing up the account.

#### Market Constraints

- Instrument: **EUR/USD**, a major FX pair with high liquidity and tight spreads.
- Market is open 24h (Mon–Fri) but **behaviour changes by session**:
  - Asia: lower volatility, more ranges.
  - London: higher volatility, breakouts and trends.
  - NY: continuation or reversals, plus US data releases.
- Need to **avoid trading around major news** (NFP, CPI, FOMC, ECB) to protect a small account.

#### Operational Constraints

- Use **Python** with open‑source tools (pandas, numpy, scikit‑learn, etc.).
- Historical data source for research: **`yfinance`**, later replaceable by broker API data.
- No assumptions of tick‑level precision or institutional infrastructure.
- Models must be simple enough to monitor, debug, and retrain as needed.
***

### 1.3 Objectives & Success Criteria



#### Primary Success Criteria (Trading Performance)

The system is considered successful if, on realistic backtests and forward tests:

- **Expectancy per trade > 0** (profitable on average).
- **Maximum drawdown** is kept below a target threshold (e.g. 15–25%).
- **Profit factor > 1.2–1.5** (total gains / total losses).
- **Risk‑adjusted returns**:
  - Sharpe or Sortino ratio above a chosen minimum (e.g. > 1.0–1.5).
- Performance is **reasonably stable across time**:
  - not coming from one lucky month or regime only.
  - holds out‑of‑sample and on unseen periods.

#### Secondary Success Criteria (ML Metrics)

Model‑level metrics are useful but not the final objective:

- Accuracy, precision, recall for trade success classification.
- Calibration of predicted probabilities.
- Stability of model performance across rolling windows.
- AUC/F1 used **only as a guide**, not as the main optimization target.

Trading metrics > ML metrics.
Models are judged by strategy performance, not just statistical scores.

***

### 1.4 Domain Understanding (Trading Concepts)


#### Forex Basics

- We trade **currency pairs**; here, **EUR/USD** (Euro vs US Dollar).
- Price expresses how many USD one Euro is worth.
- Core mechanics:
  - **Pips**: smallest price unit (0.0001 for EUR/USD).
  - **Lots**: position size (1 lot = 100k units; we use micro‑lots like 0.01).
  - **Spread**: difference between bid and ask; cost of entering a trade.
  - **Leverage**: allows controlling a larger notional position with small capital (dangerous if misused).
  - **Margin**: amount of capital locked as collateral when a trade is open.

#### Liquidity

- EUR/USD is the **most liquid pair** in FX:
  - tightest spreads,
  - lowest slippage,
  - smoother price action compared to exotics.
- High liquidity is ideal for:
  - day‑trading,
  - ML modeling (less random gaps/spikes),
  - small‑account trading.

#### Sessions & Time‑of‑Day Effects

- **Asia session**: generally low volatility, more sideways movement.
- **London session**: high liquidity and volatility, common breakouts and intraday trends.
- **NY session**: strong moves around US data; mix of continuation and reversals.
- **Late NY**: often low activity and choppy noise.

Session behaviour will be **confirmed empirically** in the Data Understanding phase and later used as features (e.g. `session`, `hour_of_day`).

#### Macro Fundamentals (Minimal Usage for Now)

- Macro drivers: interest rates, inflation data, employment data, central bank decisions.
- For this project’s first phase:
- Fundamentals are used mainly to identify **“no‑trade” windows** around high‑impact news.
- Objective is to **avoid** extremely unpredictable conditions rather than model macro directly at first.


***

### 1.5 Analytical Questions


#### Market Behaviour Questions

1. How does **volatility** (range, ATR, return magnitude) vary by hour of day and session (Asia/London/NY)?
2. Is there statistical evidence of:
   - **mean reversion** in certain hours/sessions?
   - **breakouts** after periods of volatility compression?
   - **trend persistence** intraday?
3. What are the typical intraday **range sizes** and holding periods that are feasible with small capital?

#### Modeling Questions

4. Which features (returns, volatility, candle patterns, time‑of‑day, range distance) show useful relationships with future price movement?
5. Does including **session/time features** improve predictability?
6. Can we define **trade events** where the model’s prediction is meaningful (i.e., non‑random payoff distributions)?

#### Risk & Money Management Questions

7. What stop‑loss and take‑profit distances make sense for EUR/USD on 5‑minute data given spreads and small account size?
8. How sensitive is strategy performance to:
   - risk per trade,
   - max number of trades per day,
   - session filters?

#### Strategy Questions

9. In which contexts does **mean reversion** perform better than **breakout** or **trend‑following**, and vice versa?
10. Can we identify **market regimes** (range, trend, high‑vol news, dead) using unsupervised methods (e.g. clustering, HMMs)?
11. How well do model‑based entry filters (probability thresholds) improve raw rule‑based strategies?

#### Bot & Deployment Questions

12. What is the simplest reliable architecture to:
    - fetch data in real time,
    - generate features,
    - apply models and risk logic,
    - execute trades via broker API,
    - and log everything for monitoring and retraining?


***

***


## 2) Data Understanding

### 2.1 Data Sources

#### 2.1.1 Price Data (Primary)

- For the initial research/prototyping phase:

  - Use `yfinance` to obtain:
    - 5-minute EUR/USD data for the last ~60 days (intraday structure, sessions, initial modeling).
    - 1-hour and/or daily data for 1–5 years (long-term volatility and regime analysis).

- For later production-grade modeling and backtesting:

  - Plan to switch to broker or dedicated FX historical data for:
    - 5-minute (or 1-minute) EUR/USD over 2–5+ years.

- Fields:
  - `Datetime` (index, UTC or local timezone depending on feed)
  - `Open`, `High`, `Low`, `Close`
  - `Volume` (for FX via Yahoo this is typically 0 or not meaningful → will be ignored)

This dataset will be used for:

- Session and volatility analysis (Data Understanding).
- Feature engineering (Data Preparation).
- ML model training and backtesting (Modeling & Evaluation).

> **Note:** In later phases, the Yahoo data can be replaced or augmented with broker‑specific data (e.g., OANDA/MT5) to align historical tests more closely with live trading conditions.

#### 2.1.2 1‑Minute Data (Optional / Short Window)

- Yahoo Finance provides **1‑minute data** only for relatively short lookback windows (e.g., last 7 days).
- Usage in this project:
  - **NOT** for long‑term backtesting or primary ML training (history too short, too noisy).
  - Used instead for:
    - microstructure exploration,
    - checking behaviour around entries and exits,
    - testing live data pipelines and bot execution.

#### 2.1.3 Economic Calendar / News Data (Planned)

- Purpose: filter out **high‑impact news windows** where price behaviour is extremely unstable.
- Content needed:
  - Event timestamp (UTC)
  - Event name and currency (e.g., US NFP, US CPI, ECB rate decision)
  - Impact rating (low/medium/high)
- Usage:
  - Create features like `is_high_impact_news_now`, `is_high_impact_news_next_30m`.
  - Hard “no‑trade” rule around major events for small account safety.

In early prototyping, this can be mocked or added later once the core price‑based pipeline is stable.


### 2.2 Loading the datasets and inspecting for missing values

#### 2.2.1) 5 min interval(60 days)

In [3]:
# Loading the historical price from yfinance for the EURUSD for the last 60 days at 5 min intervals

five_min = yf.download(tickers="EURUSD=X", interval="5m", period="60d")

# Setting the 'datetime' as the index

five_min.index = pd.to_datetime(five_min.index)

five_min.tail()

C:\Users\eugene.maina\AppData\Local\Temp\ipykernel_14040\1379596454.py:3: FutureWarning: YF.download() has changed argument auto_adjust default to True
  five_min = yf.download(tickers="EURUSD=X", interval="5m", period="60d")
[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,EURUSD=X,EURUSD=X,EURUSD=X,EURUSD=X,EURUSD=X
Datetime,,,,,
2025-11-28 10:25:00+00:00,1.157006,1.157006,1.156738,1.156738,0
2025-11-28 10:30:00+00:00,1.156872,1.157006,1.156872,1.156872,0
2025-11-28 10:35:00+00:00,1.156604,1.156738,1.156604,1.156604,0
2025-11-28 10:40:00+00:00,1.156470,1.156604,1.156470,1.156604,0
2025-11-28 10:45:00+00:00,1.156470,1.156470,1.156337,1.156470,0


In [4]:
five_min.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 16900 entries, 2025-09-07 23:00:00+00:00 to 2025-11-28 10:45:00+00:00
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   (Close, EURUSD=X)   16900 non-null  float64
 1   (High, EURUSD=X)    16900 non-null  float64
 2   (Low, EURUSD=X)     16900 non-null  float64
 3   (Open, EURUSD=X)    16900 non-null  float64
 4   (Volume, EURUSD=X)  16900 non-null  int64  
dtypes: float64(4), int64(1)
memory usage: 792.2 KB


#### 2.2.2) 1 min interval(7 days)

In [12]:
# Loading the historical data for 1 min intervals for the last 7 days

one_min = yf.download(tickers="EURUSD=X", interval="1m", period="7d")

one_min.index = pd.to_datetime(one_min.index)


one_min.tail()

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,EURUSD=X,EURUSD=X,EURUSD=X,EURUSD=X,EURUSD=X
Datetime,,,,,
2025-11-28 10:55:00+00:00,1.155936,1.155936,1.155936,1.155936,0
2025-11-28 10:56:00+00:00,1.155936,1.155936,1.155936,1.155936,0
2025-11-28 10:57:00+00:00,1.156069,1.156069,1.156069,1.156069,0
2025-11-28 10:58:00+00:00,1.156203,1.156203,1.156203,1.156203,0
2025-11-28 10:59:00+00:00,1.156203,1.156203,1.156203,1.156203,0


In [13]:
one_min.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 9146 entries, 2025-11-20 00:00:00+00:00 to 2025-11-28 10:59:00+00:00
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   (Close, EURUSD=X)   9146 non-null   float64
 1   (High, EURUSD=X)    9146 non-null   float64
 2   (Low, EURUSD=X)     9146 non-null   float64
 3   (Open, EURUSD=X)    9146 non-null   float64
 4   (Volume, EURUSD=X)  9146 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 428.7 KB


#### 2.2.3) 1 day interval(2 years)

In [14]:
# Loading the 1 day trading data for the last 2 years

day_data = yf.download(tickers='EURUSD=X', interval='1d', period='2y')

day_data.index = pd.to_datetime(day_data.index)

day_data.tail()

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,EURUSD=X,EURUSD=X,EURUSD=X,EURUSD=X,EURUSD=X
Date,,,,,
2025-11-24,1.150761,1.155135,1.150285,1.150788,0
2025-11-25,1.152007,1.158346,1.151212,1.152074,0
2025-11-26,1.156484,1.160227,1.154961,1.156577,0
2025-11-27,1.160214,1.161575,1.157716,1.160362,0
2025-11-28,1.156203,1.160497,1.155936,1.160093,0


In [15]:
day_data.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 521 entries, 2023-11-28 to 2025-11-28
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   (Close, EURUSD=X)   521 non-null    float64
 1   (High, EURUSD=X)    521 non-null    float64
 2   (Low, EURUSD=X)     521 non-null    float64
 3   (Open, EURUSD=X)    521 non-null    float64
 4   (Volume, EURUSD=X)  521 non-null    int64  
dtypes: float64(4), int64(1)
memory usage: 24.4 KB


#### 2.2.4) Economic news

In [16]:
# Loading the economic news data set

events_df = pd.read_csv("data/economic calendar dataset/Calender_data.csv", index_col='id')

events_df.head()

,date,time,zone,currency,importance,event,actual,forecast,previous
id,,,,,,,,,
9,01/01/2020,All Day,united states,NaN,NaN,United States - New Year's Day,NaN,NaN,NaN
9,01/01/2020,All Day,united kingdom,NaN,NaN,United Kingdom - New Year's Day,NaN,NaN,NaN
9,01/01/2020,All Day,germany,NaN,NaN,Germany - New Year's Day,NaN,NaN,NaN
9,01/01/2020,All Day,switzerland,NaN,NaN,Switzerland - New Year's Day,NaN,NaN,NaN
9,01/01/2020,All Day,italy,NaN,NaN,Italy - New Year's Day,NaN,NaN,NaN


In [17]:
events_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 663497 entries, 9 to 693
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   date        663497 non-null  object
 1   time        656887 non-null  object
 2   zone        663497 non-null  object
 3   currency    115290 non-null  object
 4   importance  115290 non-null  object
 5   event       663497 non-null  object
 6   actual      106757 non-null  object
 7   forecast    40317 non-null   object
 8   previous    106818 non-null  object
dtypes: object(9)
memory usage: 50.6+ MB


* Form loading and using the `.info()` method, all the price data seems to have no missing values with all of the columns except _Volume_ to be float datatypes. The Economic event calendar seems to have various missing values, however it may not be the case when we select out events that are to do with `USD` and `EUR` only.

In [21]:
# Selecting relevant events for the target currencies

relevant_events = events_df[events_df['currency'].isin(['EUR', 'USD'])]


relevant_events.info()


relevant_events.to_csv("data/economic calendar dataset/relevant_events.csv", index=False)

<class 'pandas.core.frame.DataFrame'>
Index: 51634 entries, 471386 to 532573
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   date        51634 non-null  object
 1   time        51634 non-null  object
 2   zone        51634 non-null  object
 3   currency    51634 non-null  object
 4   importance  51634 non-null  object
 5   event       51634 non-null  object
 6   actual      46448 non-null  object
 7   forecast    16679 non-null  object
 8   previous    46431 non-null  object
dtypes: object(9)
memory usage: 3.9+ MB


In [ ]:
relevant_events.head()



,date,time,zone,currency,importance,event,actual,forecast,previous
id,,,,,,,,,
471386,01/01/2020,20:30,united states,USD,medium,Fed's Balance Sheet,"4,173.63B",NaN,"4,165.59B"
398321,02/01/2020,08:00,austria,EUR,low,Austrian Unemployment Change (Dec),349.8K,NaN,299.5K
398324,02/01/2020,08:00,austria,EUR,low,Austrian Unemployment Rate (Dec),8.5%,NaN,7.3%
398337,02/01/2020,11:00,portugal,EUR,low,Portuguese Business Confidence (Dec),2.1,NaN,2.1
398335,02/01/2020,11:00,portugal,EUR,low,Portuguese Consumer Confidence (Dec),-7.20,NaN,-6.90


## 3) Data Preparation

## 4) Data Modelling

## 5) Evaluation

## 6) Deployment